# Build the blog data-viz JSON

Takes the scraped/curated CSVs in `../albums/{album}/{input,output}` and produces the two JSON files every chart on the blog post reads:

- `album_data_viz_{album}.json` — one record per track (Spotify's own tracklist is the source of truth for which tracks exist and their order), joined against scraped credits.
- `samples_data_viz_{album}.json` — one record per sample connection (what a track pulled from the past, what later pulled from it), re-keyed against the same canonical track list so every chart agrees on track identity and order.

(Merged from the former `join-data.ipynb` + `get-samples-timeline.ipynb` — same two outputs, one notebook, one shared set of helpers.)

In [1]:
import os
import re
import json
import pandas as pd

In [2]:
with open('consts.json', 'r') as file:
    data = json.load(file)

album = data['album']
artist = data['artist']

In [3]:
script_dir = os.path.dirname(os.path.abspath('build-viz-data.ipynb'))

### Shared helpers

In [4]:
def normalize(s):
    s = s.strip().lower().replace("\u2019", "'")
    s = re.sub(r'\(.*?\)', '', s)  # drop parenthetical suffixes (e.g. remix credits)
    s = re.sub(r'[.]+', ' ', s)       # ellipses/periods vary between sources
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def slugify(s):
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', s.lower())).strip('_')

def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_serializable(v) for v in obj]
    if pd.isna(obj):
        return None
    if hasattr(obj, 'item'):  # numpy int64/float64 -> python native
        return obj.item()
    return obj

## Part 1 — credits + audio features -> `album_data_viz_{album}.json`

The only identifier on the scraped credits is the track name, which isn't the strongest unique string — there can be little discrepancies between sources (curly quotes, remix subtitles, ellipses). `normalize()` smooths those out before joining on Spotify's track id.

In [5]:
credits_csv_file_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'credits_{album}.csv')
audio_feats_w_play_counts_csv_file_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'audio_feats_and_play_counts_{album}.csv')

df_credits = pd.read_csv(credits_csv_file_path)
df_audio = pd.read_csv(audio_feats_w_play_counts_csv_file_path)

In [6]:
df_audio["join_key"] = df_audio["track_name"].map(normalize)
df_credits["join_key"] = df_credits["track_name"].map(normalize)

# sanity check before trusting the join
unmatched = set(df_credits["join_key"]) - set(df_audio["join_key"])
if unmatched:
    print("Unmatched credit rows — check these manually:", unmatched)

In [7]:
credits_with_id = df_credits.merge(
    df_audio[["join_key", "spotify_track_id"]], on="join_key", how="left"
)

In [8]:
def get_primary(role_name, group):
    match = group[(group.role == role_name) & (group.is_primary == True)]
    return match["person"].iloc[0] if len(match) else None

In [9]:
# df_audio (Spotify's own tracklist) is the source of truth for which tracks
# exist and their order. Not every track has scraped credits (e.g. interludes) -
# those just get an empty credits list / no primary producer or mixer.
album_records = []
for _, track_row in df_audio.sort_values("track_number").iterrows():
    spotify_track_id = track_row["spotify_track_id"]
    group = credits_with_id[credits_with_id.spotify_track_id == spotify_track_id]
    album_records.append({
        "spotify_track_id": spotify_track_id,
        "track_name": track_row["track_name"],
        "track_number": track_row.get("track_number"),
        "play_count": int(track_row["play_count"]),
        "play_count_last_updated": track_row["play_count_last_updated"],
        "audio_features": {
            "energy": track_row["energy"],
            "danceability": track_row["danceability"],
            "valence": track_row["valence"],
            "tempo": track_row["tempo"],
        },
        "primary_producer": get_primary("producer", group) if len(group) else None,
        "primary_mixer": get_primary("mixer", group) if len(group) else None,
        "credits": group[["role", "role_detail", "person", "is_primary"]].to_dict("records"),
    })

album_records = make_serializable(album_records)
len(album_records)

16

In [10]:
album_output_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'album_data_viz_{album}.json')
with open(album_output_path, 'w') as f:
    json.dump(album_records, f, indent=2)

album_output_path

'/Users/binguyen/Desktop/Hip-Hop-Time-Travelers/data/src/../albums/Like Water For Chocolate/output/album_data_viz_Like Water For Chocolate.json'

## Part 2 — samples -> `samples_data_viz_{album}.json`

`get-samples.ipynb`'s scraped output never captured sample *direction* (sampled vs. was-sampled-in/remixed-in/covered-in), only the element (drums, vocals, etc.), so it can't tell past from future — the manually curated `input/{album_slug}_samples.csv` has direction + year + element, so that's the source of truth here.

Track names in that file don't always match exactly (case, remix subtitles), so every row gets re-keyed against `album_records` from Part 1 above — same canonical track identity and order as the rest of the site, no second read of the JSON we just wrote.

In [11]:
canonical_by_key = {
    normalize(t['track_name']): (t['track_name'], t['track_number'])
    for t in album_records
}

album_slug = slugify(f'{artist} {album}')
album_slug

'common_like_water_for_chocolate'

In [12]:
samples_input_path = os.path.join(
    script_dir, '..', 'albums', album, 'input', f'{album_slug}_samples.csv'
)
df_samples = pd.read_csv(samples_input_path)
df_samples.head()

,album,album_artist,album_year,track_number,track_title,track_artist,sample_type,sample_title,sample_artist,sample_year,sample_element,sample_image_url
0,Like Water for Chocolate,Common,2000,1,The Light,Common,sampled,Open Your Eyes,Bobby Caldwell,1980,Multiple Elements,https://www.whosampled.com/static/images/media...
1,Like Water for Chocolate,Common,2000,1,The Light,Common,sampled,You're Getting a Little Too Smart,Detroit Emeralds,1973,Drums,https://www.whosampled.com/static/images/media...
2,Like Water for Chocolate,Common,2000,1,The Light,Common,sampled,Track 3 (Another Batch),J Dilla,1998,Multiple Elements,https://www.whosampled.com/static/images/redes...
3,Like Water for Chocolate,Common,2000,1,The Light,Common,was sampled in,Good Flirts,Baby Keem feat. Kendrick Lamar and Momo Boyd,2026,Vocals / Lyrics,https://www.whosampled.com/static/images/media...
4,Like Water for Chocolate,Common,2000,1,The Light,Common,was sampled in,I Love Her Again,J. Cole,2026,Vocals / Lyrics,https://www.whosampled.com/static/images/media...


In [13]:
df_samples = df_samples[df_samples['album'].str.casefold() == album.casefold()].copy()
df_samples['sample_year'] = pd.to_numeric(df_samples['sample_year'], errors='coerce').astype('Int64')
df_samples = df_samples.dropna(subset=['sample_year'])

df_samples['join_key'] = df_samples['track_title'].map(normalize)

unmatched_samples = set(df_samples['join_key']) - set(canonical_by_key)
if unmatched_samples:
    print('Unmatched sample track titles — check these manually:', unmatched_samples)

df_samples['track_name'] = df_samples['join_key'].map(lambda k: canonical_by_key.get(k, (None, None))[0])
df_samples['track_number'] = df_samples['join_key'].map(lambda k: canonical_by_key.get(k, (None, None))[1])
df_samples = df_samples.dropna(subset=['track_name'])
len(df_samples)

182

In [14]:
sample_records = df_samples.rename(columns={
    'sample_element': 'element',
})[[
    'track_name', 'track_number', 'sample_title', 'sample_artist',
    'sample_year', 'sample_type', 'element', 'album_year',
]].rename(columns={'sample_type': 'direction', 'album_year': 'release_year'})
sample_records = sample_records.sort_values(['track_number', 'sample_year']).to_dict('records')
sample_records = make_serializable(sample_records)
sample_records[0]

{'track_name': 'Heat',
 'track_number': 2,
 'sample_title': 'Asiko (In a Silent Mix)',
 'sample_artist': 'Tony Allen',
 'sample_year': 1999,
 'direction': 'sampled',
 'element': 'Multiple Elements',
 'release_year': 2000}

In [15]:
samples_output_path = os.path.join(
    script_dir, '..', 'albums', album, 'output', f'samples_data_viz_{album}.json'
)
with open(samples_output_path, 'w') as f:
    json.dump(sample_records, f, indent=2)

samples_output_path

'/Users/binguyen/Desktop/Hip-Hop-Time-Travelers/data/src/../albums/Like Water For Chocolate/output/samples_data_viz_Like Water For Chocolate.json'